# Synthetic invoices

CPU only. Generates 1000 train, 100 validation and 100 test invoices (50 of them on held out templates) and publishes them as a private Hub dataset.

Secret required: `HF_TOKEN`.

In [ ]:
import os
import subprocess

REPO = "https://github.com/N-Mehdi/receipt-vlm.git"
DATASET = "N-Mehdi/french-invoices-synth"
ADAPTER = "N-Mehdi/qwen3-vl-2b-receipts-lora"
OUT = "/kaggle/working/outputs"


def run(cmd):
    """Run a shell command, stream its output and fail the notebook on error."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("command failed: " + cmd)


run("git clone --depth 1 " + REPO + " /tmp/receipt-vlm")
os.chdir("/tmp/receipt-vlm")
run("git log -1 --format=%H")
run("pip install -q -r requirements.txt && pip install -q --no-deps -e .")

In [ ]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
os.environ["HF_TOKEN"] = client.get_secret("HF_TOKEN")

In [ ]:
run("playwright install --with-deps chromium")
run(
    "python scripts/generate_invoices.py --out /kaggle/working/data/invoices"
    + " --push-to-hub "
    + DATASET
)